# Step 08C — Prepare ORIGINAL box correction tasks for CVAT (22 flagged radiographs)

**No training, no inference, no source modification.** Creates two import-ready **Ultralytics YOLO Detection** ZIPs with **original images and original labels**, plus original-box-number overlays and a traceable manifest.

- 20 internal-test images, 2 external images.
- The previously corrected 11 empty-label images are **not** included here.
- **CAUTION:** Reviewer notes `BOX 1` etc may refer to visual inspection numbering, not the original label-file line order. Numbered overlays here use **1-based YOLO file line order**. Confirm mapping with clinicians before deleting/resizing.
- Only upload images to CVAT if dataset license and institutional policy allow use of the hosted service.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from PIL import Image, ImageDraw, ImageFont
import csv, json, zipfile, hashlib, io, shutil

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Pediatric_Caries_AI')
DATA={
 'internal':ROOT/'processed_data/yolo_caries_baseline',
 'external':ROOT/'external_validation/MOPG7_66'
}
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=ROOT/'reviewer_revision'/f'Step_08C_SourceBox_CVAT_Prep_{RUN_ID}'
OUT.mkdir(parents=True,exist_ok=False)
print('Review packages will be saved in:',OUT)


In [ ]:
CASES=[
  {
    "cohort": "external",
    "image": "ibn115.jpg",
    "source_boxes": 1,
    "expert_note": "PRIORITY: inspect anomalous large source box; do not auto-delete.",
    "review_request": "Resolve No/No contradiction and document decision; Explicit adjudication of anomalous source box"
  },
  {
    "cohort": "internal",
    "image": "cate1_013.png",
    "source_boxes": 7,
    "expert_note": "ALL BOX FSLSE",
    "review_request": "Resolve No/No contradiction and document decision"
  },
  {
    "cohort": "external",
    "image": "0000847.jpg",
    "source_boxes": 1,
    "expert_note": "",
    "review_request": "Provide expert-approved full corrected label file; Explain which lesion is missing/needs edit"
  },
  {
    "cohort": "internal",
    "image": "cate1_001.png",
    "source_boxes": 3,
    "expert_note": "BOX 2 IS FALSE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate1_002.png",
    "source_boxes": 7,
    "expert_note": "BOX 1.2.3.4 IS FALSE BOX 5.6.7 TOO LARGE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate1_004.png",
    "source_boxes": 8,
    "expert_note": "BOX 1.2.3.4.6.7.8. IS FALSE BOX 5 TOO LARGE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate1_005.png",
    "source_boxes": 2,
    "expert_note": "BOX 1 FALSE . BOX 2 TOO LARGE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate1_006.png",
    "source_boxes": 4,
    "expert_note": "BOX 1.2.3 FALSE .BOX 4 TOO LARGE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate1_007.png",
    "source_boxes": 9,
    "expert_note": "BOX 1.2.3.4 IS FALSE BOX 5.6.7.8.9 TOO LARGE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate1_008.png",
    "source_boxes": 4,
    "expert_note": "BOX 1.2.3.4. IS FALSE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate1_011.png",
    "source_boxes": 8,
    "expert_note": "BOX 1.2.4.5.8 IS FALSE BOX3.6.7 TOO LARGE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate2_001.png",
    "source_boxes": 2,
    "expert_note": "BOX1 TOO LARGE BOX 2 FALSE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate2_002.png",
    "source_boxes": 5,
    "expert_note": "ALL BOX FSLSE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate2_003.png",
    "source_boxes": 5,
    "expert_note": "BOX 1.2.4.5.FALSE BOX 3 TOO LAGR",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate2_004.png",
    "source_boxes": 5,
    "expert_note": "BOX 1.2 FALSE BOX 3.4.5 TOO LARGE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate2_005.png",
    "source_boxes": 2,
    "expert_note": "BOX 1 TOO LARGE BOX 2 FALSE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate2_006.png",
    "source_boxes": 9,
    "expert_note": "BOX 1.2.4.5. TOO LARGE BOX 3 .6.7.8 FALSE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate2_007.png",
    "source_boxes": 1,
    "expert_note": "BOX IS FALSE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate2_008.png",
    "source_boxes": 5,
    "expert_note": "BOX IS FALSE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate2_009.png",
    "source_boxes": 5,
    "expert_note": "BOX IS FALSE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate2_011.png",
    "source_boxes": 10,
    "expert_note": "BOX.2.3.4.7.8.9.10 FALSE BOX 1.5.6. TOO LARGE",
    "review_request": "Provide expert-approved full corrected label file"
  },
  {
    "cohort": "internal",
    "image": "cate2_012.png",
    "source_boxes": 1,
    "expert_note": "BOX IS FALSE",
    "review_request": "Provide expert-approved full corrected label file"
  }
]
assert len(CASES)==22 and sum(c['cohort']=='internal' for c in CASES)==20 and sum(c['cohort']=='external' for c in CASES)==2
print('Original source cases: 22 (20 internal, 2 external)')


## Check all 22 inputs and coordinate geometry
The code checks original image and original YOLO label file identities, exactly matching the source box counts recorded by the expert-review tracker. It does NOT use any AI prediction to pick or edit annotations.


In [ ]:
def sha256_bytes(data):return hashlib.sha256(data).hexdigest()
def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1<<20),b''):h.update(b)
 return h.hexdigest()

def parse_label(p):
 lines=[x.strip() for x in p.read_text().splitlines() if x.strip()]
 output=[]
 for i,l in enumerate(lines,1):
  tokens=l.split()
  assert len(tokens)==5,(p,i,l)
  cid=float(tokens[0]);v=list(map(float,tokens[1:]))
  assert cid==0,(p,i,'Wrong class',cid)
  x,y,w,h=v
  assert 0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1,(p,i,v)
  assert x-w/2>=-1e-4 and x+w/2<=1+1e-4 and y-h/2>=-1e-4 and y+h/2<=1+1e-4,(p,i,'Out of frame',v)
  output.append((x,y,w,h))
 return output

AUDIT=[]
for c in CASES:
 root=DATA[c['cohort']];p=root/'images/test'/c['image'];lab=root/'labels/test'/(Path(c['image']).stem+'.txt')
 assert p.is_file(),f'Missing ORIGINAL image {p}'
 assert lab.is_file(),f'Missing ORIGINAL label {lab}'
 with Image.open(p) as im: W,H=im.size;im.verify()
 boxes=parse_label(lab)
 assert len(boxes)==c['source_boxes'],('SOURCE BOX COUNT MISMATCH — STOP, NO CVAT IMPORT',c['image'],len(boxes),c['source_boxes'])
 row=dict(**c,width=W,height=H,image_SHA256=sha256_file(p),original_label_SHA256=sha256_file(lab))
 AUDIT.append(row)
print('PASS: 22 original image-label pairs, coordinates and recorded box counts.')
print('Original source boxes:',sum(x['source_boxes'] for x in AUDIT))


## Generate full dataset archives for direct CVAT PROJECT import
Each ZIP includes exactly the original radiographs, complete original label files, `data.yaml` and `train.txt`. **DO NOT** confuse these with the corrected outputs, which will be exported from CVAT after specialist approval.


In [ ]:
FILES={}
for cohort in ['internal','external']:
 rows=[r for r in AUDIT if r['cohort']==cohort]
 name='Step08C_Internal_20_Original_GT_Import.zip' if cohort=='internal' else 'Step08C_External_2_Original_GT_Import.zip'
 dst=OUT/name
 with zipfile.ZipFile(dst,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
  z.writestr('data.yaml','path: ./\ntrain: train.txt\nnames:\n  0: caries\n')
  z.writestr('train.txt',''.join('images/train/'+r['image']+'\n' for r in rows))
  for r in rows:
   base=DATA[cohort]
   ip=base/'images/test'/r['image'];lp=base/'labels/test'/(Path(r['image']).stem+'.txt')
   z.write(ip,'images/train/'+ip.name)
   z.write(lp,'labels/train/'+lp.name)
 with zipfile.ZipFile(dst) as z:
  assert z.testzip() is None
  assert len([n for n in z.namelist() if n.startswith('images/train/')])==len(rows)
  assert len([n for n in z.namelist() if n.startswith('labels/train/')])==len(rows)
 FILES[cohort]=dst
 print('Created:',dst,'| Images:',len(rows),'| original boxes:',sum(r['source_boxes'] for r in rows))


## Numbered original-source annotation reference overlays (NOT for uploading as training images)
The numbered boxes follow 1-based **original YOLO TXT line order**, which may NOT match the clinician's handwritten `BOX #`. The clinician must verify any numbered decision. Originals are never modified.


In [ ]:
OVER=OUT/'Original_GT_Numbered_Review_Overlays';OVER.mkdir()
for r in AUDIT:
 cohort=r['cohort'];ip=DATA[cohort]/'images/test'/r['image'];lp=DATA[cohort]/'labels/test'/(Path(r['image']).stem+'.txt')
 boxes=parse_label(lp)
 with Image.open(ip) as orig:
  im=orig.convert('RGB');W,H=im.size
  d=ImageDraw.Draw(im)
  for num,(xc,yc,w,h) in enumerate(boxes,1):
   xy=((xc-w/2)*W,(yc-h/2)*H,(xc+w/2)*W,(yc+h/2)*H)
   thickness=max(2,int(min(W,H)*0.004))
   d.rectangle(xy,outline='red',width=thickness)
   tx=max(0,int(xy[0]));ty=max(0,int(xy[1])-22)
   name=f'ORIGINAL BOX #{num}'
   bbox=d.textbbox((tx,ty),name)
   d.rectangle(bbox,fill='black');d.text((tx,ty),name,fill='yellow')
  ext=im.save(OVER/f'{cohort}__{ip.stem}_ORIGINAL_BOX_NUMBERS.png')
print('Created',len(list(OVER.glob('*.png'))),'reference overlays (do not upload them as task images).')


## Save review manifest, annotations decision tracker, and export a single small audit ZIP
Original image files remain only in their two CVAT-import ZIPs in Drive. Keep the original source labels unmodified; export COMPLETE corrected labels after review.


In [ ]:
with (OUT/'Step08C_Original_GT_Source_Manifest.csv').open('w',newline='',encoding='utf-8-sig') as f:
 wr=csv.DictWriter(f,fieldnames=list(AUDIT[0]));wr.writeheader();wr.writerows(AUDIT)
with (OUT/'Step08C_Expert_Approval_Tracker.csv').open('w',newline='',encoding='utf-8-sig') as f:
 cols=['cohort','image','source_boxes','expert_note','review_request','operator_action','corrected_box_count','expert1_confirmed','expert2_confirmed','consensus_final','review_date','signed_by']
 wr=csv.DictWriter(f,fieldnames=cols);wr.writeheader()
 for r in AUDIT:
  wr.writerow(dict(cohort=r['cohort'],image=r['image'],source_boxes=r['source_boxes'],expert_note=r['expert_note'],review_request=r['review_request'],expert1_confirmed='PENDING',expert2_confirmed='PENDING',consensus_final='PENDING'))
README="""STEP 08C / ORIGINAL-BOX ANNOTATION CORRECTION (22 images)\n\nA) CVAT: create NEW project Pediatric_Caries_SourceBox_Corrections_v1, one label: caries (Rectangle).\nB) Project -> Actions -> Import dataset -> Ultralytics YOLO Detection 1.0; import Internal_20 ZIP, then External_2 ZIP.\nC) Ensure original boxes visible and image count is correct (20 and 2). STOP if missing.\nD) Compare raw images and numbered original overlays; NOTE numbers are YOLO file line order, and cannot be assumed to match experts handwritten box indices.\nE) Review all boxes: KEEP / RESIZE / DELETE / ADD / UNCERTAIN; never auto-delete. Record consensus with two specialists.\nF) Each task -> Actions -> Export task dataset -> Ultralytics YOLO Detection 1.0; Save images disabled.\nG) Save exported full corrected archives as Step08C_Internal_20_Corrected_GT.zip and Step08C_External_2_Corrected_GT.zip.\nH) Share the TWO exported ZIPs and signed tracker with ChatGPT for preflight BEFORE sensitivity evaluation.\nNo weights, training or model inference involved. Original GT remains intact.\n"""
(OUT/'README_NEXT_STEPS.txt').write_text(README)
qap=OUT/'Step08C_Review_Audit_Only.zip'
with zipfile.ZipFile(qap,'w',zipfile.ZIP_DEFLATED,compresslevel=5) as z:
 for filename in ['README_NEXT_STEPS.txt','Step08C_Original_GT_Source_Manifest.csv','Step08C_Expert_Approval_Tracker.csv']:
  z.write(OUT/filename,filename)
 for img in sorted(OVER.glob('*.png')):z.write(img,'Original_GT_Numbered_Review_Overlays/'+img.name)
print('DONE. In Google Drive:',OUT)
for f in FILES.values():print('CVAT DATASET',f.name,'size bytes',f.stat().st_size)
print('REVIEW AUDIT',qap.name)
